# 08 RL Mini Project TODO

## 목표
MountainCar에서 tabular RL과 naive DQN의 한계를 확인하고, replay buffer와 target network를 직접 넣어보며 DQN 안정화의 필요성을 이해한다.

## 학생이 직접 채울 TODO
- `discretize_state`
- Q-learning update
- SARSA update
- naive DQN의 핵심 TD update
- replay buffer의 `push()` / `sample()`
- target network 복사


## MountainCar 한눈에 보기
- 상태(state): `[position, velocity]`
- 행동(action): `0=왼쪽`, `1=가속 안 함`, `2=오른쪽`
- 보상(reward): 매 step마다 `-1`
- 종료: 정상 도달 또는 `200 step`
- 핵심 난점: 왕복하며 속도를 누적해야 정상에 도달할 수 있다.

## 오늘 던질 질문
- 연속 상태를 Q-table에 바로 넣을 수 없는 이유는?
- discretization을 하면 어떤 정보가 사라지는가?
- naive DQN이 왜 불안정할까?
- replay buffer는 무엇을 완화하고, target network는 무엇을 고정할까?


In [ ]:
import random

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display

torch.set_num_threads(1)
plt.style.use("seaborn-v0_8")


In [ ]:
SEED = 7

TABULAR_CONFIG = {
    "n_position_bins": 20,
    "n_velocity_bins": 20,
    "episodes": 1500,
    "max_steps": 200,
    "alpha": 0.15,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "avg_window": 20,
    "log_every": 100,
}

NAIVE_DQN_CONFIG = {
    "episodes": 80,
    "max_steps": 200,
    "hidden_dim": 32,
    "lr": 1e-3,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "avg_window": 20,
    "log_every": 20,
    "loss_log_window": 200,
}

STABLE_DQN_CONFIG = {
    "episodes": 100,
    "max_steps": 200,
    "hidden_dim": 32,
    "lr": 5e-4,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "avg_window": 20,
    "log_every": 20,
    "loss_log_window": 50,
    "buffer_capacity": 5000,
    "batch_size": 32,
    "learn_start": 200,
    "train_frequency": 4,
    "target_sync_steps": 200,
}

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def make_env(seed=None):
    env = gym.make("MountainCar-v0")
    if seed is not None:
        env.action_space.seed(seed)
    return env

def linear_epsilon(episode_idx, total_episodes, start, end):
    progress = episode_idx / max(total_episodes - 1, 1)
    return max(end, start - (start - end) * progress)

def moving_average(values, window=20):
    values = np.asarray(values, dtype=np.float32)
    if len(values) == 0:
        return values
    if len(values) < window:
        return values.copy()
    weights = np.ones(window, dtype=np.float32) / window
    return np.convolve(values, weights, mode="valid")

def plot_reward_curves(reward_dict, title, avg_window=20):
    plt.figure(figsize=(10, 4))
    for label, rewards in reward_dict.items():
        rewards = list(rewards)
        plt.plot(rewards, alpha=0.25, label=f"{label} raw")
        smooth = moving_average(rewards, avg_window)
        start_idx = avg_window - 1 if len(rewards) >= avg_window else 0
        plt.plot(range(start_idx, start_idx + len(smooth)), smooth, linewidth=2, label=f"{label} MA({avg_window})")
    plt.title(title)
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

def plot_loss_curves(loss_dict, title, avg_window=50):
    plt.figure(figsize=(10, 4))
    for label, losses in loss_dict.items():
        losses = list(losses)
        if not losses:
            continue
        smooth = moving_average(losses, avg_window)
        start_idx = avg_window - 1 if len(losses) >= avg_window else 0
        plt.plot(range(start_idx, start_idx + len(smooth)), smooth, linewidth=2, label=label)
    plt.title(title)
    plt.xlabel("Gradient step")
    plt.ylabel("Loss")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

def build_summary_row(name, state_repr, result, stability, note):
    recent_avg = float(np.mean(result["rewards"][-20:])) if result["rewards"] else np.nan
    return {
        "algorithm": name,
        "state_repr": state_repr,
        "last_reward": round(float(result["rewards"][-1]), 2),
        "recent_20_avg": round(recent_avg, 2),
        "goals_reached": result["goals_reached"],
        "first_success_episode": result["first_success_episode"] if result["first_success_episode"] is not None else "-",
        "stability": stability,
        "one_line_interpretation": note,
    }

def show_logs(label, result):
    print(f"[{label}]")
    display(pd.DataFrame(result["logs"]))

seed_everything(SEED)
print("Seed:", SEED)


In [ ]:
env = make_env(SEED)
obs_low = env.observation_space.low
obs_high = env.observation_space.high
print("Observation space:", env.observation_space)
print("Action space size:", env.action_space.n)
print("Observation low:", obs_low)
print("Observation high:", obs_high)
print("Goal position:", env.unwrapped.goal_position)
env.close()

tabular_bins = np.array([TABULAR_CONFIG["n_position_bins"], TABULAR_CONFIG["n_velocity_bins"]])

def discretize_state(state, obs_low=obs_low, obs_high=obs_high, bins=tabular_bins):
    state = np.array(state, dtype=np.float32)
    # TODO 1. state를 [0, 1) 비율로 바꾸세요.
    ratios = None
    # TODO 2. 범위를 벗어나지 않게 clip 하세요.
    clipped = None
    # TODO 3. bin index로 바꾸세요.
    indices = None
    return tuple(indices)

def select_action_from_q(q_table, discrete_state, epsilon, env):
    if random.random() < epsilon:
        return env.action_space.sample()
    # TODO 4. greedy action을 고르세요.
    greedy_action = None
    return int(greedy_action)

sample_env = make_env(SEED)
sample_state, _ = sample_env.reset(seed=SEED)
print("Sample state:", sample_state)
print("Discretized state:", discretize_state(sample_state))
sample_env.close()


In [ ]:
def train_tabular(method="q_learning", config=TABULAR_CONFIG, seed=SEED):
    assert method in {"q_learning", "sarsa"}
    seed_everything(seed)
    env = make_env(seed)
    bins = np.array([config["n_position_bins"], config["n_velocity_bins"]])
    q_table = np.zeros((bins[0], bins[1], env.action_space.n), dtype=np.float32)
    rewards, logs = [], []
    goals_reached = 0
    first_success_episode = None

    for episode in range(1, config["episodes"] + 1):
        state, _ = env.reset(seed=seed + episode)
        discrete_state = discretize_state(state, obs_low, obs_high, bins)
        epsilon = linear_epsilon(episode - 1, config["episodes"], config["epsilon_start"], config["epsilon_end"])
        if method == "sarsa":
            action = select_action_from_q(q_table, discrete_state, epsilon, env)
        episode_reward = 0.0

        for _ in range(config["max_steps"]):
            if method == "q_learning":
                action = select_action_from_q(q_table, discrete_state, epsilon, env)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            next_discrete_state = discretize_state(next_state, obs_low, obs_high, bins)
            episode_reward += reward

            if method == "q_learning":
                # TODO 5. Q-learning target을 계산하세요.
                td_target = None
                # TODO 6. Q-table을 업데이트하세요.
                q_table[discrete_state][action] += config["alpha"] * (td_target - q_table[discrete_state][action])
            else:
                if done:
                    # TODO 7. 종료 상태의 SARSA target을 계산하고 업데이트하세요.
                    td_target = None
                    q_table[discrete_state][action] += config["alpha"] * (td_target - q_table[discrete_state][action])
                else:
                    next_action = select_action_from_q(q_table, next_discrete_state, epsilon, env)
                    # TODO 8. 종료가 아닐 때 SARSA target을 계산하고 업데이트하세요.
                    td_target = None
                    q_table[discrete_state][action] += config["alpha"] * (td_target - q_table[discrete_state][action])
                    action = next_action

            discrete_state = next_discrete_state
            if done:
                if next_state[0] >= env.unwrapped.goal_position:
                    goals_reached += 1
                    if first_success_episode is None:
                        first_success_episode = episode
                break

        rewards.append(episode_reward)
        if episode % config["log_every"] == 0:
            recent_avg = float(np.mean(rewards[-config["avg_window"] :]))
            logs.append({"episode": episode, "reward": round(float(episode_reward), 2), "recent_avg": round(recent_avg, 2), "epsilon": round(float(epsilon), 3), "goals": goals_reached})
            print(f"[{method}] episode={episode:4d}, reward={episode_reward:6.1f}, recent_avg={recent_avg:7.2f}, epsilon={epsilon:5.3f}, goals={goals_reached}")

    env.close()
    return {"q_table": q_table, "rewards": rewards, "logs": logs, "goals_reached": goals_reached, "first_success_episode": first_success_episode}


In [ ]:
# TODO를 채운 뒤 실행하세요.
q_learning_result = train_tabular("q_learning")
sarsa_result = train_tabular("sarsa")

plot_reward_curves({"Q-learning": q_learning_result["rewards"], "SARSA": sarsa_result["rewards"]}, "Tabular RL on MountainCar", avg_window=TABULAR_CONFIG["avg_window"])
show_logs("Q-learning", q_learning_result)
show_logs("SARSA", sarsa_result)


## Tabular 해석 메모
- 최근 20 episode 평균 reward는 언제부터 움직였는가?
- Q-learning과 SARSA는 어떤 차이를 보였는가?
- bin 수를 바꾸면 어떤 변화가 예상되는가?


## 2. naive DQN
이제 raw continuous state를 그대로 넣는 DQN으로 넘어간다.
단, 이번 버전은 replay buffer와 target network를 일부러 빼 둔 naive한 형태다.


In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )

    def forward(self, x):
        # TODO 9. 순전파 결과를 반환하세요.
        return None

def state_to_tensor(state):
    return torch.tensor(state, dtype=torch.float32).unsqueeze(0)

def select_action_dqn(state, epsilon, q_network, env):
    if random.random() < epsilon:
        return env.action_space.sample()
    state_tensor = state_to_tensor(state)
    with torch.no_grad():
        q_values = q_network(state_tensor)
    # TODO 10. 가장 큰 Q-value의 action index를 고르세요.
    action = None
    return int(action)


In [ ]:
def train_naive_dqn(config=NAIVE_DQN_CONFIG, seed=SEED):
    seed_everything(seed)
    env = make_env(seed)
    online_net = QNetwork(env.observation_space.shape[0], env.action_space.n, config["hidden_dim"])
    optimizer = optim.Adam(online_net.parameters(), lr=config["lr"])
    loss_fn = nn.MSELoss()
    rewards, losses, logs = [], [], []
    goals_reached = 0
    first_success_episode = None

    for episode in range(1, config["episodes"] + 1):
        state, _ = env.reset(seed=seed + episode)
        epsilon = linear_epsilon(episode - 1, config["episodes"], config["epsilon_start"], config["epsilon_end"])
        episode_reward = 0.0

        for _ in range(config["max_steps"]):
            action = select_action_dqn(state, epsilon, online_net, env)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            state_tensor = state_to_tensor(state)
            next_state_tensor = state_to_tensor(next_state)
            q_values = online_net(state_tensor)
            # TODO 11. 현재 action의 Q-value를 고르세요.
            predicted_q = None

            with torch.no_grad():
                next_q_values = online_net(next_state_tensor)
                # TODO 12. 다음 상태의 최대 Q-value를 계산하세요.
                max_next_q = None
                # TODO 13. done 여부에 따라 target을 계산하세요.
                td_target = None

            target_tensor = torch.tensor(td_target, dtype=torch.float32)
            # TODO 14. loss를 계산하세요.
            loss = None
            optimizer.zero_grad()
            loss.backward()
            # TODO 15. optimizer step을 실행하세요.

            losses.append(float(loss.item()))
            episode_reward += reward
            state = next_state
            if done:
                if next_state[0] >= env.unwrapped.goal_position:
                    goals_reached += 1
                    if first_success_episode is None:
                        first_success_episode = episode
                break

        rewards.append(episode_reward)
        if episode % config["log_every"] == 0:
            recent_avg = float(np.mean(rewards[-config["avg_window"] :]))
            recent_losses = losses[-config["loss_log_window"] :]
            loss_mean = float(np.mean(recent_losses)) if recent_losses else np.nan
            loss_std = float(np.std(recent_losses)) if recent_losses else np.nan
            logs.append({"episode": episode, "reward": round(float(episode_reward), 2), "recent_avg": round(recent_avg, 2), "epsilon": round(float(epsilon), 3), "loss_mean": round(loss_mean, 4), "loss_std": round(loss_std, 4), "goals": goals_reached})
            print(f"[naive_dqn] episode={episode:3d}, reward={episode_reward:6.1f}, recent_avg={recent_avg:7.2f}, loss_mean={loss_mean:8.4f}, loss_std={loss_std:8.4f}, goals={goals_reached}")

    env.close()
    return {"online_net": online_net, "rewards": rewards, "losses": losses, "logs": logs, "goals_reached": goals_reached, "first_success_episode": first_success_episode}


In [ ]:
# TODO를 채운 뒤 실행하세요.
naive_dqn_result = train_naive_dqn()

plot_reward_curves({"Naive DQN": naive_dqn_result["rewards"]}, "Naive DQN Reward Curve", avg_window=NAIVE_DQN_CONFIG["avg_window"])
plot_loss_curves({"Naive DQN": naive_dqn_result["losses"]}, "Naive DQN Loss Curve", avg_window=50)
show_logs("Naive DQN", naive_dqn_result)


## naive DQN 해석 메모
- reward는 거의 움직였는가, 아니면 바닥에 머물렀는가?
- loss가 크게 흔들렸는가?
- 왜 같은 네트워크로 target까지 만들면 위험한가?


## 3. replay buffer와 target network 추가
이번 단계에서는 `replay only`와 `replay + target`을 비교해 본다.


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def push(self, state, action, reward, next_state, done):
        transition = (np.array(state, dtype=np.float32), int(action), float(reward), np.array(next_state, dtype=np.float32), float(done))
        # TODO 16. capacity 전에는 append, 꽉 차면 현재 position에 덮어쓰세요.
        pass
        # TODO 17. circular index를 갱신하세요.

    def sample(self, batch_size):
        # TODO 18. random mini-batch를 뽑으세요.
        batch = None
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.array(states), dtype=torch.float32),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(np.array(next_states), dtype=torch.float32),
            torch.tensor(dones, dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buffer)


In [ ]:
def train_stabilized_dqn(use_target_network=True, config=STABLE_DQN_CONFIG, seed=SEED):
    seed_everything(seed)
    env = make_env(seed)
    online_net = QNetwork(env.observation_space.shape[0], env.action_space.n, config["hidden_dim"])
    target_net = QNetwork(env.observation_space.shape[0], env.action_space.n, config["hidden_dim"])
    target_net.load_state_dict(online_net.state_dict())
    optimizer = optim.Adam(online_net.parameters(), lr=config["lr"])
    loss_fn = nn.MSELoss()
    replay_buffer = ReplayBuffer(config["buffer_capacity"])
    rewards, losses, logs = [], [], []
    goals_reached = 0
    first_success_episode = None
    total_steps = 0
    variant_name = "replay+target" if use_target_network else "replay_only"

    for episode in range(1, config["episodes"] + 1):
        state, _ = env.reset(seed=seed + episode)
        epsilon = linear_epsilon(episode - 1, config["episodes"], config["epsilon_start"], config["epsilon_end"])
        episode_reward = 0.0

        for _ in range(config["max_steps"]):
            action = select_action_dqn(state, epsilon, online_net, env)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            # TODO 19. transition을 replay buffer에 저장하세요.
            total_steps += 1
            episode_reward += reward

            enough_samples = len(replay_buffer) >= config["batch_size"]
            can_learn = total_steps >= config["learn_start"]
            learn_now = total_steps % config["train_frequency"] == 0

            if enough_samples and can_learn and learn_now:
                # TODO 20. mini-batch를 sample 하세요.
                states, actions, rewards_batch, next_states, dones = None
                predicted_q = online_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    # TODO 21. target network 사용 여부에 따라 next_source를 고르세요.
                    next_source = None
                    max_next_q = next_source(next_states).max(dim=1).values
                    targets = rewards_batch + config["gamma"] * max_next_q * (1.0 - dones)
                loss = loss_fn(predicted_q, targets)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                losses.append(float(loss.item()))

            if use_target_network and total_steps % config["target_sync_steps"] == 0:
                # TODO 22. online -> target으로 가중치를 복사하세요.
                pass

            state = next_state
            if done:
                if next_state[0] >= env.unwrapped.goal_position:
                    goals_reached += 1
                    if first_success_episode is None:
                        first_success_episode = episode
                break

        rewards.append(episode_reward)
        if episode % config["log_every"] == 0:
            recent_avg = float(np.mean(rewards[-config["avg_window"] :]))
            recent_losses = losses[-config["loss_log_window"] :]
            loss_mean = float(np.mean(recent_losses)) if recent_losses else np.nan
            loss_std = float(np.std(recent_losses)) if recent_losses else np.nan
            logs.append({"episode": episode, "reward": round(float(episode_reward), 2), "recent_avg": round(recent_avg, 2), "epsilon": round(float(epsilon), 3), "loss_mean": round(loss_mean, 4), "loss_std": round(loss_std, 4), "goals": goals_reached})
            print(f"[{variant_name}] episode={episode:3d}, reward={episode_reward:6.1f}, recent_avg={recent_avg:7.2f}, loss_mean={loss_mean:8.4f}, loss_std={loss_std:8.4f}, goals={goals_reached}")

    env.close()
    return {"online_net": online_net, "target_net": target_net, "rewards": rewards, "losses": losses, "logs": logs, "goals_reached": goals_reached, "first_success_episode": first_success_episode}


In [ ]:
# TODO를 채운 뒤 실행하세요.
replay_dqn_result = train_stabilized_dqn(use_target_network=False)
target_dqn_result = train_stabilized_dqn(use_target_network=True)

plot_reward_curves({"Naive DQN": naive_dqn_result["rewards"], "Replay only": replay_dqn_result["rewards"], "Replay + Target": target_dqn_result["rewards"]}, "DQN Variant Reward Comparison", avg_window=20)
plot_loss_curves({"Naive DQN": naive_dqn_result["losses"], "Replay only": replay_dqn_result["losses"], "Replay + Target": target_dqn_result["losses"]}, "DQN Variant Loss Comparison", avg_window=50)
show_logs("Replay only", replay_dqn_result)
show_logs("Replay + Target", target_dqn_result)


## replay buffer / target network 해석 메모
- replay buffer만 넣었을 때 무엇이 바뀌었는가?
- target network를 추가하니 loss 진동은 어떻게 달라졌는가?
- 이번 주차 목표가 “성능 최고점”이 아니라 “안정화 필요성 이해”인 이유는 무엇인가?


In [ ]:
comparison_rows = [
    build_summary_row("Q-learning", "20x20 discretized state", q_learning_result, "직접 작성", ""),
    build_summary_row("SARSA", "20x20 discretized state", sarsa_result, "직접 작성", ""),
    build_summary_row("Naive DQN", "raw continuous state", naive_dqn_result, "직접 작성", ""),
    build_summary_row("DQN + Replay", "raw continuous state", replay_dqn_result, "직접 작성", ""),
    build_summary_row("DQN + Replay + Target", "raw continuous state", target_dqn_result, "직접 작성", ""),
]
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)


## 학생 기록 템플릿
| Algorithm | State representation | Episode reward | Recent 20 avg | Goals reached | Stability | One-line interpretation |
|---|---|---:|---:|---:|---|---|
| Q-learning |  |  |  |  |  |  |
| SARSA |  |  |  |  |  |  |
| Naive DQN |  |  |  |  |  |  |
| DQN + Replay |  |  |  |  |  |  |
| DQN + Replay + Target |  |  |  |  |  |  |

## 자주 나는 오류와 힌트
- discretization index가 bin 범위를 넘는다면 `np.clip(..., 0.0, 0.999999)`을 다시 확인한다.
- Gymnasium은 `terminated`, `truncated`를 둘 다 확인해야 한다.
- DQN에서 `gather`를 쓸 때는 `actions.unsqueeze(1)`이 필요하다.
- replay buffer 길이가 `batch_size`보다 짧을 때 `sample()` 하면 에러가 난다.
- target network는 선언만 하면 끝이 아니라 주기적으로 복사해야 한다.

## 시간이 부족할 때
1. 튜닝보다 로그 해석을 우선한다.
2. 최소한 `naive DQN`과 `Replay + Target`은 비교한다.
3. 여유가 있으면 bin 수나 episode 수를 바꿔 추가 실험한다.
